# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a two-stage task: first classification (is this page declining or not — is_declining_label), then that classification's probability feeds into a ranking/scoring step (rank pages by how likely/severely they're declining, so the team knows which to review first). The final output is a rank, but the classification step underneath is what makes the ranking meaningful, it's not an arbitrary sort, it's a sort backed by a learned probability.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


**What I would predict:** `is_declining_label` (whether `trend_direction == "down"`).

**Where it comes from:** This is a defined rule, not a truly future observed outcome, `trend_direction` is calculated by comparing the last 30 days of impressions to the previous 30 days within the same 90-day snapshot. So it tells us whether a page is currently showing a decline pattern, not whether it will decline in the future.

**Honest limitation:** A stronger version of this target would use the full warehouse's daily table to build a true future-window label (features from a prior period predicting an outcome in a later, non-overlapping period). I attempted this using DuckDB + the Hugging Face warehouse release, but ran into environment/authentication issues I couldn't resolve today — I plan to revisit this before the Week 4 lane-confirmation deadline.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I would defend **Precision@50** as my main metric. Content team can only realistically review a limited number of pages each cycle. Precision@50 tells us directly, of the top 50 pages the model recommends reviewing first, how many were actually correct? That matches exactly how the output will be used, unlike a generic metric like accuracy or ROC AUC, which measures overall separation but doesn't reflect the team's real capacity.

In my pipeline run: baseline precision@50 = 0.24 (~12 correct out of 50), random forest precision@50 = 0.74 (~37 correct out of 50). "Good" means beating the baseline by a meaningful margin at this specific K — not just having a higher AUC in the abstract.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nOne row = one content item (a single page), identified by content_id.")
print(f"Unique content_id count: {df['content_id'].nunique()}")

df[["content_id", "client_id", "content_type", "impressions_90d", "trend_direction"]].head()

Shape: 30000 rows, 44 columns

One row = one content item (a single page), identified by content_id.
Unique content_id count: 30000


,content_id,client_id,content_type,impressions_90d,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,down


One row = **one content item (a single page)**, uniquely identified by `content_id`. The dataframe has 30,000 rows and 30,000 unique `content_id` values, confirming no duplicates each page appears exactly once, with its own 90-day aggregated metrics (impressions, clicks, sessions, etc.) and its own `trend_direction` label.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (if-statement) can only check conditions one at a time, in a rigid order, e.g. "IF days_since_last_update >= 180 AND impressions_90d >= 500 THEN flag it." But my random forest's feature importance shows the real signal comes from **many features weighted together** (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`, and more), each contributing a different amount, and their *combination* matters more than any single threshold. Writing that by hand as nested if-statements would require guessing every possible combination and its right threshold, and get more fragile and error-prone with every extra condition added. That's exactly what my baseline vs. random forest comparison shows, the rule-based baseline (0.627 AUC) is a reasonable start, but the model that can weigh 52 features simultaneously (0.750 AUC) captures a pattern too tangled for a hand-written rule to reach.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.